# Coffee Shop Sales Analysis


## Problem
Identify peak hours, weekday/weekend differences, seasonal trends, popular drinks, spending patterns, and price elasticity to support data-driven decisions for a coffee shop manager.

## Dataset
Daily Coffee Transactions (Kaggle, accessed 08 April 2026)

## 1. Import Libraries

In [19]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [ ]:
# Create output folders and define a helper for saving figures
from pathlib import Path

FIGURES_DIR = Path('figures')
FIGURES_DIR.mkdir(exist_ok=True)

def save_current_figure(filename):
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / filename, dpi=300, bbox_inches='tight')


## 2. Load Data

In [ ]:
# Load the dataset from the repo structure
possible_paths = [
    Path('data/Coffe_sales.csv'),
    Path('Coffe_sales.csv')
]

for csv_path in possible_paths:
    if csv_path.exists():
        df = pd.read_csv(csv_path)
        print(f"Loaded dataset from: {csv_path}")
        break
else:
    raise FileNotFoundError('Could not find Coffe_sales.csv in data/ or project root.')


## 3. Display First Few Rows

In [21]:
print("First 5 rows of the dataset:")
df.head()

First 5 rows of the dataset:


,hour_of_day,cash_type,money,coffee_name,Time_of_Day,Weekday,Month_name,Weekdaysort,Monthsort,Date,Time
0,10,card,38.7,Latte,Morning,Fri,Mar,5,3,2024-03-01,10:15:50.520000
1,12,card,38.7,Hot Chocolate,Afternoon,Fri,Mar,5,3,2024-03-01,12:19:22.539000
2,12,card,38.7,Hot Chocolate,Afternoon,Fri,Mar,5,3,2024-03-01,12:20:18.089000
3,13,card,28.9,Americano,Afternoon,Fri,Mar,5,3,2024-03-01,13:46:33.006000
4,13,card,38.7,Latte,Afternoon,Fri,Mar,5,3,2024-03-01,13:48:14.626000


In [22]:
# Check missing values
print("\nMissing values:")
print(df.isnull().sum())


Missing values:
hour_of_day    0
cash_type      0
money          0
coffee_name    0
Time_of_Day    0
Weekday        0
Month_name     0
Weekdaysort    0
Monthsort      0
Date           0
Time           0
dtype: int64


## 4. Data Cleaning & Feature Engineering

In [23]:
# Check basic info
print("Dataset info:")
df.info()

Dataset info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3547 entries, 0 to 3546
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   hour_of_day  3547 non-null   int64  
 1   cash_type    3547 non-null   object 
 2   money        3547 non-null   float64
 3   coffee_name  3547 non-null   object 
 4   Time_of_Day  3547 non-null   object 
 5   Weekday      3547 non-null   object 
 6   Month_name   3547 non-null   object 
 7   Weekdaysort  3547 non-null   int64  
 8   Monthsort    3547 non-null   int64  
 9   Date         3547 non-null   object 
 10  Time         3547 non-null   object 
dtypes: float64(1), int64(3), object(7)
memory usage: 304.9+ KB


In [24]:
# Drop rows with missing critical values (if any)
df = df.dropna(subset=['money', 'coffee_name', 'cash_type'])

In [25]:
# Rename for convenience
df.rename(columns={'hour_of_day': 'hour', 'Weekday': 'weekday', 'Month_name': 'month'}, inplace=True)

In [26]:
# Display cleaned data
print("\nCleaned data preview:")
df.head()


Cleaned data preview:


,hour,cash_type,money,coffee_name,Time_of_Day,weekday,month,Weekdaysort,Monthsort,Date,Time
0,10,card,38.7,Latte,Morning,Fri,Mar,5,3,2024-03-01,10:15:50.520000
1,12,card,38.7,Hot Chocolate,Afternoon,Fri,Mar,5,3,2024-03-01,12:19:22.539000
2,12,card,38.7,Hot Chocolate,Afternoon,Fri,Mar,5,3,2024-03-01,12:20:18.089000
3,13,card,28.9,Americano,Afternoon,Fri,Mar,5,3,2024-03-01,13:46:33.006000
4,13,card,38.7,Latte,Afternoon,Fri,Mar,5,3,2024-03-01,13:48:14.626000


## 5. Basic Descriptive Analysis

In [27]:
# Total transactions, average price, etc.
total_transactions = len(df)
avg_price = df['money'].mean()
print(f"Total transactions: {total_transactions}")
print(f"Average transaction value: ${avg_price:.2f}")

Total transactions: 3547
Average transaction value: $31.65


In [28]:
# Most popular drinks
top_drinks = df['coffee_name'].value_counts().head(5)
print("\nTop 5 most popular drinks:")
print(top_drinks)


Top 5 most popular drinks:
coffee_name
Americano with Milk    809
Latte                  757
Americano              564
Cappuccino             486
Cortado                287
Name: count, dtype: int64


In [ ]:
# Price distribution
plt.figure(figsize=(10,5))
plt.hist(df['money'], bins=20, edgecolor='black', alpha=0.7)
plt.title('Distribution of Transaction Values')
plt.xlabel('Amount ($)')
plt.ylabel('Frequency')
plt.grid(True, linestyle='--', alpha=0.5)
save_current_figure('transaction_values_distribution.png')
plt.show()


## 6. Time-based Analysis

In [30]:
# 6.1 Hourly sales volume
hourly_sales = df.groupby('hour').size().reset_index(name='count')
peak_hour = hourly_sales.loc[hourly_sales['count'].idxmax(), 'hour']
peak_qty = hourly_sales['count'].max()
print(f"Peak hour: {peak_hour}:00 with {peak_qty} transactions")

Peak hour: 10:00 with 328 transactions


In [ ]:
# Plot
plt.figure(figsize=(12,5))
plt.bar(hourly_sales['hour'], hourly_sales['count'], color='skyblue')
plt.title('Number of Transactions by Hour of Day')
plt.xlabel('Hour (24h)')
plt.ylabel('Number of Transactions')
plt.xticks(range(0,24,2))
plt.grid(axis='y', linestyle='--', alpha=0.7)
save_current_figure('hourly_transactions.png')
plt.show()


In [ ]:
# 6.2 Weekday vs Weekend
print("Unique values in 'weekday' column:", df['weekday'].unique())

weekend_set = {'Sat', 'Sun', 'Saturday', 'Sunday'}

if 'Weekdaysort' in df.columns:
    df['day_type'] = df['Weekdaysort'].apply(lambda x: 'Weekend' if int(x) >= 6 else 'Weekday')
else:
    df['day_type'] = df['weekday'].apply(lambda x: 'Weekend' if x in weekend_set else 'Weekday')

weekday_weekend = df.groupby('day_type').size().reindex(['Weekday', 'Weekend'])
weekday_qty = int(weekday_weekend['Weekday'])
weekend_qty = int(weekday_weekend['Weekend'])
weekday_ratio = weekday_qty / weekend_qty

print(f"Weekday transactions: {weekday_qty}, Weekend: {weekend_qty}")
print(f"Weekday is {weekday_ratio:.1f} times busier than weekend")


In [ ]:
# 6.3 Monthly trend
if {'month', 'Monthsort'}.issubset(df.columns):
    monthly_sales = (
        df.groupby(['Monthsort', 'month'])
        .size()
        .reset_index(name='count')
        .sort_values('Monthsort')
    )
    peak_row = monthly_sales.loc[monthly_sales['count'].idxmax()]
    peak_month = peak_row['month']
    monthly_max = int(peak_row['count'])
    print(f"Peak month: {peak_month} with {monthly_max} transactions")

    plt.figure(figsize=(10,5))
    plt.plot(monthly_sales['month'], monthly_sales['count'], marker='o', linestyle='-', color='green')
    plt.title('Monthly Transaction Trend')
    plt.xlabel('Month')
    plt.ylabel('Number of Transactions')
    plt.grid(True)
    save_current_figure('monthly_transaction_trend.png')
    plt.show()


## 7. Behavioral Economics Extensions

In [ ]:
# 7.1 Average spending by hour
hourly_avg_price = df.groupby('hour', as_index=False)['money'].mean().sort_values('hour')
peak_avg_row = hourly_avg_price.loc[hourly_avg_price['money'].idxmax()]
peak_avg_hour = int(peak_avg_row['hour'])
peak_avg_value = float(peak_avg_row['money'])

print("Average spending by hour:")
print(hourly_avg_price)
print(f"Highest average spending occurs at {peak_avg_hour}:00 (${peak_avg_value:.2f}).")

plt.figure(figsize=(12,5))
plt.plot(hourly_avg_price['hour'], hourly_avg_price['money'], marker='o', linestyle='-', color='orange')
plt.title('Average Transaction Value by Hour')
plt.xlabel('Hour')
plt.ylabel('Average Amount ($)')
plt.grid(True)
save_current_figure('average_spending_by_hour.png')
plt.show()


In [ ]:
# 7.2 Weekend vs Weekday average spending
weekday_avg_price = df.groupby('day_type')['money'].mean().reindex(['Weekday', 'Weekend'])
spending_gap = weekday_avg_price['Weekday'] - weekday_avg_price['Weekend']

if abs(spending_gap) < 0.2:
    spending_message = 'Average spending is very similar on weekdays and weekends.'
elif spending_gap > 0:
    spending_message = 'Average spending is slightly higher on weekdays.'
else:
    spending_message = 'Average spending is slightly higher on weekends.'

print(f"Weekday avg spending: ${weekday_avg_price['Weekday']:.2f}, Weekend avg: ${weekday_avg_price['Weekend']:.2f}")
print(spending_message)


In [ ]:
# 7.3 Sales volume by price range
price_bins = [15, 20, 30, 50]
price_labels = ['15-20', '20-30', '30-50']
df['price_range'] = pd.cut(df['money'], bins=price_bins, labels=price_labels, right=False)
price_range_sales = df['price_range'].value_counts().sort_index()
top_price_range = price_range_sales.idxmax()

print("Sales volume by price range:")
print(price_range_sales)
print(f"The busiest price range is {top_price_range}.")

plt.figure(figsize=(8,5))
price_range_sales.plot(kind='bar', color='lightcoral')
plt.title('Demand by Price Range')
plt.xlabel('Price Range ($)')
plt.ylabel('Number of Transactions')
plt.grid(axis='y', linestyle='--', alpha=0.7)
save_current_figure('price_range_demand.png')
plt.show()


In [37]:
# 7.4 Top drinks: price vs popularity
top5_drinks = top_drinks.index.tolist()
drink_stats = df[df['coffee_name'].isin(top5_drinks)].groupby('coffee_name').agg(
    avg_price=('money', 'mean'),
    total_sales=('coffee_name', 'count')
).sort_values('total_sales', ascending=False)
print("Top drinks - price and sales:")
print(drink_stats)

Top drinks - price and sales:
                     avg_price  total_sales
coffee_name                                
Americano with Milk  30.594710          809
Latte                35.502378          757
Americano            25.975638          564
Cappuccino           35.883004          486
Cortado              25.731220          287


In [ ]:
# Scatter plot: price vs popularity
plt.figure(figsize=(10,6))
sns.scatterplot(data=drink_stats, x='avg_price', y='total_sales', size='total_sales', sizes=(100,500), legend=False)
for i, row in drink_stats.iterrows():
    plt.text(row['avg_price']+0.5, row['total_sales']+5, i, fontsize=9)
plt.title('Drink Price vs Popularity')
plt.xlabel('Average Price ($)')
plt.ylabel('Total Sales')
plt.grid(True, linestyle='--', alpha=0.5)
save_current_figure('drink_price_vs_popularity.png')
plt.show()


## 8. Key Insights Summary

In [ ]:
print("\n" + "="*60)
print("KEY INSIGHTS SUMMARY")
print("="*60)
print(f"1. Peak hour: {peak_hour}:00 with {peak_qty} transactions.")
print(f"2. Weekday transactions: {weekday_qty}, Weekend: {weekend_qty} -> Weekday {weekday_ratio:.1f}x busier.")
print(f"3. Peak month: {peak_month} with {monthly_max} transactions.")
print(f"4. Most popular drinks: {top_drinks.index[0]} ({top_drinks.iloc[0]}), {top_drinks.index[1]} ({top_drinks.iloc[1]}).")
print(f"5. Highest average spending occurs at {peak_avg_hour}:00 (${peak_avg_value:.2f}).")
print(f"6. Weekday avg spending: ${weekday_avg_price['Weekday']:.2f}; Weekend avg spending: ${weekday_avg_price['Weekend']:.2f}.")
print(f"   Interpretation: {spending_message}")
print(f"7. Price range {top_price_range} has the highest sales volume.")
print("="*60)
